In [ ]:
#part of the code please refer to https://github.com/AI4Finance-Foundation/FinRL
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

data prepare and index calculation

In [ ]:
import pandas as pd
import yfinance as yf

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl import config_tickers
from finrl.config import INDICATORS
from finrl.config import *
import itertools

In [ ]:
!pip install pandas_market_calendars

In [ ]:
TRAIN_START_DATE = '2024-03-31'
TRADE_END_DATE = '2025-03-31'
aapl_df_yf = yf.download(tickers = "aapl", start=TRAIN_START_DATE, end=TRADE_END_DATE)

In [ ]:
# aapl_df_finrl = YahooDownloader(start_date = TRAIN_START_DATE,
#                                 end_date = TRAIN_END_DATE,
#                                 ticker_list = ['aapl']).fetch_data()

In [ ]:
config_tickers.DOW_30_TICKER


In [ ]:
TRAIN_START_DATE = '2024-03-31'
TRAIN_END_DATE = '2025-03-31'
TRADE_START_DATE = '2025-03-31'
TRADE_END_DATE = '2025-04-15'

In [ ]:
df_raw = YahooDownloader(start_date = TRAIN_START_DATE,
                     end_date = TRADE_END_DATE,
                     ticker_list = config_tickers.DOW_30_TICKER).fetch_data()

In [ ]:
fe = FeatureEngineer(use_technical_indicator=True,
                     tech_indicator_list = INDICATORS,
                     use_vix=True,
                     use_turbulence=True,
                     user_defined_feature = False)

processed = fe.preprocess_data(df_raw)

In [ ]:
list_ticker = processed["tic"].unique().tolist()
list_date = list(pd.date_range(processed['date'].min(),processed['date'].max()).astype(str))
combination = list(itertools.product(list_date,list_ticker))

processed_full = pd.DataFrame(combination,columns=["date","tic"]).merge(processed,on=["date","tic"],how="left")
processed_full = processed_full[processed_full['date'].isin(processed['date'])]
processed_full = processed_full.sort_values(['date','tic'])

processed_full = processed_full.fillna(0)

In [ ]:
processed_full.head()

In [ ]:
train = data_split(processed_full, TRAIN_START_DATE,TRAIN_END_DATE)
trade = data_split(processed_full, TRADE_START_DATE,TRADE_END_DATE)
train = train.rename(columns={'tic': 'ticker'})
print(len(train))
print(len(trade))

In [ ]:
train.to_csv('train_data.csv')
trade.to_csv('trade_data.csv')